# Extended Data Figure 6 — Temporal evolution of L2G confidence and evidence

Two-panel figure showing how L2G gene prioritisation improves over time:

**Top panel:** Mean maximum L2G score per gene-disease pair by year (± 95 % CI)  
**Bottom panel:** Number of unique genes per year stratified by supporting evidence type
(PAV only | PAV + molQTL coloc | molQTL coloc only | Neither)

**Source notebook:** `chapters/02-analysis/03-coloc-l2g/19_temporal_l2g_improvements.ipynb`  
**Data:** `data/intermediate_files/list_of_prioritised_genes_per_CS_with_year_nfe_maf.parquet`,
`data/intermediate_files/qualifying_credible_sets/`


## Setup


In [ ]:
from gentropy.common.session import Session
from pyspark.sql import functions as f

In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

## Paths


In [ ]:
path_to_intermediate_data_folder = str(paper.DERIVED) + "/"

l2g_with_year_path = path_to_intermediate_data_folder + "prioritised_genes_annotated"
qualifying_disease_cs_path = path_to_intermediate_data_folder + "qualifying_credible_sets"
qualifying_measurement_cs_path = path_to_intermediate_data_folder + "qualifying_measurement_credible_sets"

## Load L2G data filtered to qualifying disease credible sets


In [ ]:
l2g_full = session.spark.read.parquet(l2g_with_year_path)

qd_cs = session.spark.read.parquet(qualifying_disease_cs_path).select("studyLocusId").cache()
qm_cs = session.spark.read.parquet(qualifying_measurement_cs_path).select("studyLocusId").cache()
qcs = qd_cs.union(qm_cs).distinct().cache()
print(f"Qualifying disease CSs:     {qd_cs.count():,}")
print(f"Qualifying measurement CSs: {qm_cs.count():,}")
print(f"Total qualifying CSs:       {qcs.count():,}")

# Disease + measurement — used for bottom panel (unique gene counts)
l2g_dm = l2g_full.join(qcs, on="studyLocusId", how="inner")
l2g_exploded_dm = l2g_dm.withColumn("diseaseId", f.explode("diseaseIds"))
l2g_pandas = l2g_exploded_dm.toPandas()
print(f"Disease+measurement pandas: {len(l2g_pandas):,} rows | {l2g_pandas['geneId'].nunique():,} unique genes")

# Disease only — used for top panel (mean max L2G score)
l2g_d = l2g_full.join(qd_cs, on="studyLocusId", how="inner")
l2g_pandas_disease = l2g_d.toPandas()
print(
    f"Disease-only pandas:        {len(l2g_pandas_disease):,} rows | {l2g_pandas_disease['geneId'].nunique():,} unique genes"
)

## Top panel — Mean maximum L2G score per gene-disease pair by year


In [ ]:
import pandas as pd
import numpy as np

# Top panel uses disease-only data (mean max L2G score per unique gene)
years_list = sorted(l2g_pandas_disease["year"].unique())  # include all years incl. 2025
yearly_results = []

for target_year in years_list:
    data_up = l2g_pandas_disease[l2g_pandas_disease["year"] <= target_year]
    max_scores = data_up.groupby("geneId")["score"].max()
    n = len(max_scores)
    mean_s = max_scores.mean()
    se = max_scores.std() / np.sqrt(n)
    yearly_results.append({"year": target_year, "n": n, "mean_score": mean_s, "se_mean": se})

yearly_stats = pd.DataFrame(yearly_results)
print(yearly_stats)

## Bottom panel — Gene evidence categories by year


In [ ]:
import pandas as pd
import numpy as np

l2g_copy = l2g_pandas.copy()
l2g_copy["_VEP"] = pd.to_numeric(l2g_copy.get("VEP", 0), errors="coerce").fillna(0) >= 1
l2g_copy["_eQTL_coloc"] = pd.to_numeric(l2g_copy.get("eQTL_coloc", 0), errors="coerce").fillna(0) >= 1
l2g_copy["_pQTL_coloc"] = pd.to_numeric(l2g_copy.get("pQTL_coloc", 0), errors="coerce").fillna(0) >= 1

# Count unique genes per evidence category (cumulative up to each year)
years_bottom = list(range(2015, 2025))
rows = []
for y in years_bottom:
    df_up = l2g_copy[l2g_copy["year"] <= y]
    genes_all = set(df_up["geneId"].unique())
    genes_vep = set(df_up.loc[df_up["_VEP"], "geneId"].unique())
    genes_coloc = set(df_up.loc[df_up["_eQTL_coloc"] | df_up["_pQTL_coloc"], "geneId"].unique())
    rows.append(
        {
            "year": y,
            "PAV only": len(genes_vep - genes_coloc),
            "PAV + molQTL": len(genes_vep & genes_coloc),
            "molQTL only": len(genes_coloc - genes_vep),
            "Others": len(genes_all - (genes_vep | genes_coloc)),
        }
    )

counts_df = pd.DataFrame(rows).set_index("year")
print(counts_df.tail())

## Extended Data Figure 6 — Combined plot


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(7, 9))

# ── Top panel: mean maximum L2G score per gene, 2007–2025 ─────────────────
top_data = yearly_stats[yearly_stats["year"] >= 2007]
years_int = top_data["year"].astype(int).values
ci95 = 1.96 * top_data["se_mean"].values
means = top_data["mean_score"].values

ax_top.errorbar(
    years_int,
    means,
    yerr=ci95,
    fmt="o",
    color="steelblue",
    ecolor="steelblue",
    capsize=3,
    lw=1.5,
    ms=5,
    label="Mean L2G Score (95% CI)",
)
ax_top.set_title("Mean Maximum Score per Gene by Year", fontsize=10, fontweight="bold", loc="left")
ax_top.set_ylabel("Mean L2G Score")
ax_top.set_xlabel("")
ax_top.set_ylim(0.58, 0.66)
ax_top.set_xticks(years_int)
ax_top.tick_params(axis="x", rotation=45, labelsize=8)
ax_top.legend(fontsize=9)
ax_top.grid(axis="y", linestyle="--", alpha=0.4)
ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)

# ── Bottom panel: stacked bar — unique genes by evidence category ──────────
cat_colors = {
    "PAV only": "#2196F3",
    "PAV + molQTL": "#FF9800",
    "molQTL only": "#4CAF50",
    "Others": "#9E9E9E",
}
cat_labels = {
    "PAV only": "Supported by PAV",
    "PAV + molQTL": "Supported by PAV and molQTLs",
    "molQTL only": "Supported by molQTLs only",
    "Others": "Others",
}

x = counts_df.index.astype(int)
bottom = np.zeros(len(counts_df))
totals = counts_df.sum(axis=1).values

for col, color in cat_colors.items():
    vals = counts_df[col].values.astype(float)
    ax_bot.bar(x, vals, bottom=bottom, color=color, label=cat_labels[col])
    for xi, b, v, tot in zip(x, bottom, vals, totals):
        if tot > 0 and v / tot >= 0.05:
            ax_bot.text(
                xi,
                b + v / 2,
                f"{v / tot * 100:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=7,
                fontweight="bold",
            )
    bottom = bottom + vals

ax_bot.set_title("Number of genes per year by supporting evidence", fontsize=10, fontweight="bold", loc="left")
ax_bot.set_ylabel("Number of unique genes")
ax_bot.set_xlabel("Year")
ax_bot.set_xticks(x)
ax_bot.tick_params(axis="x", rotation=45, labelsize=8)
ax_bot.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax_bot.legend(title="Category", fontsize=8, title_fontsize=9)
ax_bot.spines["top"].set_visible(False)
ax_bot.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(f"{figure_dir}/extended_figure_6.pdf", dpi=300, bbox_inches="tight")
plt.show()